# SiteSight M5: LoRA fine-tuning of Florence-2-base (GPU)

Self-contained (code embedded, no GitHub token). Trains on the **train** split, picks the epoch with
the lowest **validation** loss (a rule fixed in `configs/train.toml`), evaluates that epoch on
validation with the same harness as the zero-shot baselines, and prints the comparison.
The test split is not used here. Set **Runtime > Change runtime type > T4 GPU**, then Run all.

In [9]:
!python -m sitesight.baselines.run --model florence --adapter /content/run/epoch_4 --annotations {ANN} --images {IMG} --split test --allow-test --device cuda --save-every 50 --out {PRED}/ft_test.json
!python -m sitesight.baselines.run --model florence --annotations {ANN} --images {IMG} --split test --allow-test --device cuda --save-every 50 --out {PRED}/zs_base_test.json
!python -m sitesight.baselines.run --model owlv2 --annotations {ANN} --images {IMG} --split test --allow-test --device cuda --save-every 50 --out {PRED}/owl_test.json
!python -m sitesight.eval.run --annotations {ANN} --split test --common-images --predictions {PRED}/ft_test_grounding.json {PRED}/ft_test_regions.json {PRED}/zs_base_test_grounding.json {PRED}/zs_base_test_regions.json {PRED}/owl_test.json --out reports/colab/test_final.json

Loading weights: 100% 665/665 [00:00<00:00, 4708.45it/s]
50/214 images, 532 predictions, 1.2 s/image
100/214 images, 1054 predictions, 1.2 s/image
150/214 images, 1524 predictions, 1.1 s/image
200/214 images, 2015 predictions, 1.1 s/image
214/214 images, 2192 predictions, 1.1 s/image
Loading weights: 100% 665/665 [00:00<00:00, 5111.16it/s]
50/214 images, 606 predictions, 1.4 s/image
100/214 images, 1220 predictions, 1.3 s/image
150/214 images, 1786 predictions, 1.2 s/image
200/214 images, 2338 predictions, 1.2 s/image
214/214 images, 2525 predictions, 1.2 s/image
preprocessor_config.json: 100% 425/425 [00:00<00:00, 2.08MB/s]
config.json: 100% 414/414 [00:00<00:00, 1.90MB/s]
tokenizer_config.json: 100% 1.10k/1.10k [00:00<00:00, 3.46MB/s]
vocab.json: 100% 1.06M/1.06M [00:00<00:00, 44.0MB/s]
merges.txt: 100% 525k/525k [00:00<00:00, 117MB/s]
added_tokens.json: 100% 67.0/67.0 [00:00<00:00, 312kB/s]
special_tokens_map.json: 100% 121/121 [00:00<00:00, 673kB/s]

model.safetensors: downloading 

In [1]:
import base64, io, os, zipfile
B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(B64))).extractall("/content/sitesight")
print(sorted(os.listdir("/content/sitesight")))

['README.md', 'configs', 'pyproject.toml', 'reports', 'scripts', 'sitesight']


In [10]:
!cd /content && zip -qr m5_outputs.zip run/train_summary.json run/experiment_log.csv run/BEST_EPOCH.txt run/epoch_4 sitesight/reports/colab sitesight_data/predictions/ft_val_grounding.json sitesight_data/predictions/ft_val_regions.json sitesight_data/predictions/ft_test_grounding.json sitesight_data/predictions/ft_test_regions.json && ls -l m5_outputs.zip && unzip -l m5_outputs.zip | tail -15

-rw-r--r-- 1 root root 12202338 Nov 23 04:21 m5_outputs.zip
     5192  2026-11-23 03:58   run/epoch_4/README.md
 13005400  2026-11-23 03:58   run/epoch_4/adapter_model.safetensors
        0  2026-11-23 04:21   sitesight/reports/colab/
    11624  2026-11-23 04:07   sitesight/reports/colab/ft_val.json
    40477  2026-11-23 04:21   sitesight/reports/colab/test_final.json
    51283  2026-11-23 04:07   sitesight/reports/colab/compare_val.json
     1857  2026-11-23 04:07   sitesight/reports/colab/ft_val.md
     3903  2026-11-23 04:07   sitesight/reports/colab/compare_val.md
     3381  2026-11-23 04:21   sitesight/reports/colab/test_final.md
   190456  2026-11-23 04:07   sitesight_data/predictions/ft_val_grounding.json
   255240  2026-11-23 04:07   sitesight_data/predictions/ft_val_regions.json
   203876  2026-11-23 04:12   sitesight_data/predictions/ft_test_grounding.json
   273545  2026-11-23 04:12   sitesight_data/predictions/ft_test_regions.json
---------                     -------
 1404

In [2]:
%cd /content/sitesight
%pip -q install -e . transformers scipy peft
# Colab preinstalls an old torchao that peft refuses to import alongside; we do not use it.
%pip -q uninstall -y torchao
import torch
assert torch.cuda.is_available(), "switch the runtime to a GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

/content/sitesight
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sitesight (pyproject.toml) ... done
torch 2.14.1+cu130 | GPU: Tesla T4


In [11]:
from google.colab import files; files.download('/content/m5_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
DATA = "/content/sitesight_data"
!python scripts/download_data.py --data-dir {DATA} --min-free-gb 3
!python -m sitesight.data.via_parse --via {DATA}/extracted/via_region_data_final.json \
    --images {DATA}/extracted/UAVdata --out {DATA}/derived/annotations.json
!python -m sitesight.data.caption_fields --annotations {DATA}/derived/annotations.json \
    --out {DATA}/derived/annotations_fields.json
!python -m sitesight.data.splits --annotations {DATA}/derived/annotations_fields.json \
    --splits reports/m2/splits.json --out {DATA}/derived/annotations_curated.json
ANN, IMG = f"{DATA}/derived/annotations_curated.json", f"{DATA}/extracted/UAVdata"

dataset ready: /content/sitesight_data/extracted
images:                 1431
regions in source:      8428
annotations kept:       8428
regions dropped:        0
clipped:                6
duplicate:              12
empty_caption:          1
zero_regions:           19
annotations 8428: parsed 8396, unparsed 31, empty 1; coverage 0.9963
  bad_target:are: 2
  glued_captions: 15
  multiple_clauses: 9
  no_resource: 3
  unknown_predicate:piles: 1
  unknown_predicate:rebars: 1
applied splits to 1431 images: {'train': 1002, 'val': 215, 'test': 214}


In [4]:
# Quick smoke test first (4 training images, about a minute): fails fast if anything is wrong.
!python -m sitesight.train.lora --config configs/train.toml --annotations {ANN} --images {IMG} \
    --out-dir /content/smoke --device cuda --limit-images 4

train samples 70, val samples 25
processor_config.json: 100% 2.26k/2.26k [00:00<00:00, 7.80MB/s]
preprocessor_config.json: 100% 603/603 [00:00<00:00, 2.89MB/s]
config.json: 100% 2.39k/2.39k [00:00<00:00, 8.46MB/s]
tokenizer_config.json: 100% 198k/198k [00:00<00:00, 292MB/s]
vocab.json: 100% 798k/798k [00:00<00:00, 35.2MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 94.2MB/s]
tokenizer.json: 100% 3.75M/3.75M [00:00<00:00, 186MB/s]
added_tokens.json: 100% 22.4k/22.4k [00:00<00:00, 60.4MB/s]
special_tokens_map.json: 100% 147k/147k [00:00<00:00, 308MB/s]

model.safetensors: downloading bytes:   0% 0.00/463M [00:00<?, ?B/s]
model.safetensors: downloading bytes:   8% 35.2M/463M [00:01<00:09, 45.8MB/s, 1.94MB/s  ]
model.safetensors: downloading bytes:  18% 85.5M/463M [00:01<00:04, 88.3MB/s, 6.82MB/s  ]
model.safetensors: downloading bytes:  22% 100M/463M [00:01<00:04, 83.8MB/s, 8.84MB/s  ] 
model.safetensors: downloading bytes:  67% 310M/463M [00:02<00:00, 221MB/s, 25.9MB/s  ]
model.safetensor

In [5]:
# Training: caches vision features once, then trains LoRA on the language model; logs every epoch.
!python -m sitesight.train.lora --config configs/train.toml --annotations {ANN} --images {IMG} \
    --out-dir /content/run --device cuda

train samples 10247, val samples 1920
Loading weights: 100% 665/665 [00:00<00:00, 4543.70it/s]
caching vision features for 1203 images on cuda...
  cached vision features 100/1203 (8s)
  cached vision features 200/1203 (15s)
  cached vision features 300/1203 (22s)
  cached vision features 400/1203 (29s)
  cached vision features 500/1203 (36s)
  cached vision features 600/1203 (43s)
  cached vision features 700/1203 (50s)
  cached vision features 800/1203 (57s)
  cached vision features 900/1203 (64s)
  cached vision features 1000/1203 (72s)
  cached vision features 1100/1203 (79s)
  cached vision features 1200/1203 (86s)
  cached vision features 1203/1203 (86s)
trainable parameters: 3,244,032 of 234,688,000 (1.38%)
epoch 0 (pretrained): val loss {'all': 5.314352667062434, 'region': 6.305017216853712, 'grounding': 3.9337895175234356}
/content/sitesight/sitesight/train/lora.py:277: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consid

In [6]:
import json
best = open("/content/run/BEST_EPOCH.txt").read().split()[0]
print("best epoch by validation loss:", best)
PRED = f"{DATA}/predictions"
!mkdir -p {PRED}
!python -m sitesight.baselines.run --model florence --adapter /content/run/epoch_{best} \
    --annotations {ANN} --images {IMG} --split val --device cuda --save-every 20 --out {PRED}/ft_val.json

best epoch by validation loss: 4
Loading weights: 100% 665/665 [00:00<00:00, 4935.31it/s]
20/215 images, 195 predictions, 1.1 s/image
40/215 images, 393 predictions, 1.1 s/image
60/215 images, 538 predictions, 1.0 s/image
80/215 images, 725 predictions, 1.0 s/image
100/215 images, 926 predictions, 1.0 s/image
120/215 images, 1154 predictions, 1.0 s/image
140/215 images, 1336 predictions, 1.0 s/image
160/215 images, 1490 predictions, 1.0 s/image
180/215 images, 1673 predictions, 1.0 s/image
200/215 images, 1891 predictions, 1.0 s/image
215/215 images, 2041 predictions, 1.0 s/image


In [7]:
!mkdir -p reports/colab
!python -m sitesight.eval.run --annotations {ANN} --split val \
    --predictions {PRED}/ft_val_grounding.json {PRED}/ft_val_regions.json --out reports/colab/ft_val.json
!python -m sitesight.eval.run --from-metrics reports/m4/zero_shot_val.json reports/colab/ft_val.json \
    --out reports/colab/compare_val.json

## Grounding on `val` (percent; IoU 0.5 unless stated)

| Model | Protocol | mAP@0.5 | mAP@0.75 | Recall@0.5 | Recall@0.75 | ECE | Images |
| --- | --- | --- | --- | --- | --- | --- | --- |
| florence-2-base-lora-epoch_4-grounding | given_phrases | 39.2 | 15.9 | 46.6 | 28.6 | 0.432 | 215 |

### Recall@0.5 by object size, crowding and region kind

| Model | small | medium | large | sparse | mid-crowd | crowded | single | group |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| florence-2-base-lora-epoch_4-grounding | 35.3 (n=241) | 51.3 (n=437) | 48.1 (n=424) | 54.3 (n=258) | 47.4 (n=641) | 34.0 (n=203) | 59.9 (n=524) | 34.4 (n=578) |

### Per-class AP@0.5 (given_phrases protocol)

| Class | n GT | florence-2-base-lora-epoch_4-grounding |
| --- | --- | --- |
| concrete | 30 | 55.0 |
| concrete_pillar | 20 | 11.6 |
| concrete_pipe | 14 | 30.8 |
| concrete_structure | 1 | 100.0 |
| concrete_truck | 9 | 36.6 |
| container | 23 | 10.5 |
| crane_truck | 26 | 60.3 |
| drilling_rig | 

In [8]:
import json
print(open("/content/run/experiment_log.csv").read())
s = json.load(open("/content/run/train_summary.json"))
print({k: s[k] for k in ("best_epoch", "best_val_loss", "pretrained_val_loss", "trainable_parameters",
                          "total_parameters", "n_train_samples", "n_val_samples", "total_seconds", "gpu")})
print("METRICS_JSON_BEGIN")
print(json.dumps(json.load(open("reports/colab/ft_val.json")), separators=(",", ":"), sort_keys=True))
print("METRICS_JSON_END")

epoch,step,train_loss,lr,seconds,val_loss_all,val_loss_region,val_loss_grounding
0,0,,0.0,0.0,5.31435,6.30502,3.93379
1,641,1.65594,9.396851762554198e-05,316.9,1.3649,0.4689,2.61356
2,1282,1.12478,7.007731183536631e-05,316.3,1.3152,0.41117,2.57503
3,1923,1.03899,3.7720461761065655e-05,316.5,1.28129,0.37515,2.54407
4,2564,0.98802,1.0541306965722263e-05,316.3,1.27512,0.36275,2.54657
5,3205,0.96447,0.0,316.7,1.2765,0.36336,2.54903

{'best_epoch': 4, 'best_val_loss': 1.275121290097648, 'pretrained_val_loss': 5.314352667062434, 'trainable_parameters': 3244032, 'total_parameters': 234688000, 'n_train_samples': 10247, 'n_val_samples': 1920, 'total_seconds': 1763.3, 'gpu': 'Tesla T4'}
METRICS_JSON_BEGIN
[{"ground_truth":{"annotations_in_split":1111,"evaluated":1102,"excluded_duplicate":2,"excluded_empty_caption":1,"excluded_unparsed":6,"images":215},"images_evaluated":215,"model":"florence-2-base-lora-epoch_4-grounding","model_info":{"adapter":"/content/run/epoch_4","decoding":"greedy","device